# TTR record-type merge in SPA

This example follows the record-type merge discussed in *TTR at the SPA*. We construct two categorical TTR types, merge them symbolically, then encode the result as a Semantic Pointer Architecture (SPA) hypervector.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from pyttr.categorical import BType, Fun, PType, Pred, RecType
from pyttr.spa import Codebook, Encoder

## Symbolic TTR merge

The notation `c_man : man(x)` is a dependent field. In `pyttr`, it is a function over `x` together with the dependency label `x`.

In [ ]:
Ind = BType("Ind")
man = Pred("man", [Ind])
run = Pred("run", [Ind])

man_at_x = (Fun("v", Ind, PType(man, ["v"])), ["x"])
run_at_x = (Fun("v", Ind, PType(run, ["v"])), ["x"])

man_type = RecType({"x": Ind, "c_man": man_at_x})
run_type = RecType({"x": Ind, "c_run": run_at_x})
merged_type = man_type.merge(run_type)

assert set(merged_type.comps.labels()) == {"x", "c_man", "c_run"}
merged_type.show()

In [ ]:
from pyttr.utils import show_latex

display(show_latex(man_type))
display(show_latex(run_type))
display(show_latex(merged_type))

## SPA encoding

`Codebook` assigns stable random hypervectors to labels, type names, predicates, and variables. `Encoder` binds each record label to its field encoding and superposes the resulting bindings. The result is approximate: we compare cosine similarities rather than vector equality.

In [ ]:
encoder = Encoder(Codebook(dimensions=10_000, seed=7))
merged_pointer = encoder.encode(merged_type)

scores = {
    "man component": encoder.similarity(merged_pointer, encoder.encode(man_type)),
    "run component": encoder.similarity(merged_pointer, encoder.encode(run_type)),
    "unrelated {y : Ind}": encoder.similarity(
        merged_pointer, encoder.encode(RecType({"y": Ind}))
    ),
}
scores

## Comparing VSA models and dimensions

TorchHD calls the hypervector algebra selection `vsa`; `Codebook` forwards this keyword unchanged. The experiment below compares binary spatter codes (`BSC`), multiply-add-permute vectors (`MAP`), and holographic reduced representations (`HRR`) at several dimensionalities.

In [ ]:
results = []
for vsa in ("BSC", "MAP", "HRR"):
    for dimensions in (512, 2_048, 10_000):
        trial = Encoder(Codebook(dimensions=dimensions, vsa=vsa, seed=7))
        merged = trial.encode(merged_type)
        man_similarity = trial.similarity(merged, trial.encode(man_type))
        run_similarity = trial.similarity(merged, trial.encode(run_type))
        unrelated_similarity = trial.similarity(
            merged, trial.encode(RecType({"y": Ind}))
        )
        assert man_similarity > unrelated_similarity
        assert run_similarity > unrelated_similarity
        results.append({
            "vsa": vsa,
            "dimensions": dimensions,
            "man component": round(man_similarity, 3),
            "run component": round(run_similarity, 3),
            "unrelated": round(unrelated_similarity, 3),
        })

results

Each row is a separate seeded codebook, so the comparison isolates the VSA model and dimensionality. The merged type should have a substantially higher similarity to both of its components than to the unrelated type. This is the first approximate SPA invariant; later examples can build on it for cleanup, memory, and question answering.

In [ ]:
encoder.codebook._pointers.keys()